# INTRODUCTION

In [45]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer as tfv
from sklearn.model_selection import train_test_split as tts
from sklearn.linear_model import LogisticRegression as lr
from sklearn.metrics import accuracy_score as ac, confusion_matrix as cm, classification_report as cr
import warnings
warnings.filterwarnings("ignore")

# DATASET OVERVIEW (Dataset inspection)

In [46]:
df = pd.read_csv("spam.csv")
df["category"] = df["Category"].apply(lambda x: 1 if x == "spam" else 0)
df = df.drop(["Category"], axis=1)
df = df[["Message", "category"]]
df.tail()

,Message,category
5567,This is the 2nd time we have tried 2 contact u...,1
5568,Will ü b going to esplanade fr home?,0
5569,"Pity, * was in mood for that. So...any other s...",0
5570,The guy did some bitching but I acted like i'd...,0
5571,Rofl. Its true to its name,0


In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5157 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Message   5157 non-null   object
 1   category  5157 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 120.9+ KB


# DATA PREPROCESSING

In [47]:
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

In [48]:
df.isnull().sum()

,0
Message,0
category,0


In [50]:
category_counts = df['category'].value_counts()
print(category_counts)

category
0    4516
1     641
Name: count, dtype: int64


In [51]:
x = df["Message"]
y = df["category"]

# TF-IDF Vectorization

In [52]:
vectorizer = tfv()
x = vectorizer.fit_transform(x)

In [53]:
x_train,x_test,y_train, y_test = tts(x,y, test_size=0.2, random_state=42)

# MODELING

In [54]:
model = lr(random_state=42)
model.fit(x_train, y_train)

LogisticRegression(random_state=42)

# EVALUATION

In [55]:
predict = model.predict(x_test)
print(ac(y_test, predict))
print(cm(y_test, predict))
print(cr(y_test, predict))

0.9563953488372093
[[894   2]
 [ 43  93]]
              precision    recall  f1-score   support

           0       0.95      1.00      0.98       896
           1       0.98      0.68      0.81       136

    accuracy                           0.96      1032
   macro avg       0.97      0.84      0.89      1032
weighted avg       0.96      0.96      0.95      1032



# No Overfitting because Model has 97% train Accuracy and 96% test Accuracy -- Good Balance

# Understand how the model is able to think that a particular message is a spam or not.

Since we use TF-IDF and LogisticRegression, the model was able to learn some important words and text pattern

Spam words like: words in caps lock(e.g FREE,WIN,CASH,URGENT,CLAIM),football match fixtures,phone number


# WORD IMPORTANCE ANALYSIS

In [56]:
feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]
word_importance = pd.DataFrame({"word": feature_names, "importance": coefficients})
word_importance

,word,importance
0,00,0.527586
1,000,0.994484
2,000pes,0.000000
3,008704050406,0.080897
4,0089,0.090820
...,...,...
8704,zouk,0.000000
8705,zyada,0.000000
8706,èn,0.000000
8707,ú1,0.127099


In [57]:
word_importance.sort_values(
    by="importance",
    ascending=False
).head(20)

,word,importance
7982,txt,4.179897
1824,call,3.761420
3369,free,3.519061
7640,text,3.085322
7311,stop,3.002735
5115,mobile,2.839483
8592,www,2.768357
2063,claim,2.734949
6438,reply,2.715448
7802,to,2.689962


# This are list of top Spam Words

the HIGHER the number in the importance column the bigger the probability of being a spam

In [58]:
word_importance.sort_values(
    by="importance",
    ascending=True
).head(20)

,word,importance
4964,me,-2.322391
5250,my,-2.200672
7666,that,-1.614759
4241,it,-1.476281
3680,gt,-1.474645
4789,lt,-1.466830
1849,can,-1.450312
1790,but,-1.423458
5534,ok,-1.403324
3974,how,-1.327527


# This are list of top Non-Spam Words

the LOWER the number in the importance column the bigger the probability of being a Non-spam

Because Model cannot understand raw text we have to convert it using Vectorizer

# *CUSTOM MESSAGE TESTING* (NOW WE TEST THE MODEL MANUALLY BY INPUTTING MESSAGES OURSELF)

In [61]:
message = ["text me tomorrow to claim your reward"]
messages = vectorizer.transform(message)
pred = model.predict(messages)
#print(pred)
if pred[0] == 1:
    print("Spam")
else:
    print("Non-Spam")

Spam


Message is a spam, it consist of two spam words like "text" and "reward" and one Non-spam word like "me"

In [62]:
mess = ["call me now please"]
messs = vectorizer.transform(mess)
pre = model.predict(messs)
if pre[0] == 1:
    print("Spam")
else:
    print("Non-Spam")
#print(pre)

Non-Spam


Message is Non-Spam, it consist of one spam word (call) and one Non-spam (me)

# FINDINGS
--Spam messages contain strong identifiable keywords that TF-IDF was able to identify  using Vectorizer

--LogisticRegression was used for Classification and it performs strongly in text classification

--Precision for spam detection was very high while Recall showed some spam messages were still missed

# CONCLUSION
Model perform excellently in classification with a very high accuracy both for training and testing(No Overfitting) and precision in catching spam messages was very high though some spam messages still escaped detection. Model also perform well with manual testing.